# Daniel Paredes Valverde

## 1. Task 1 - Improve the evaluation function with new heuristics

Our project have heavily relied in the `HeatMap` approach. With this approach, we have made every important object to make a `scent` or `heat footprint` that our agent can identify from far away.

This way, it can anticipate ghost movements and have a sight on food on every step. Additionally, it has granted us the possibility of easily sealing off dangerous zones, like the initial `cage` of the ghosts.

This `HeatMap` is also good for visual representation, as one can print it out and watch how the numbers fluctuate and expand between the movements, making it easier to debug and adjust. The map is a grid or matrix that uses -1 to denote walls and rewards positive numbers (food scents) and punishes negative numbers (ghost heat footprints).

### 1.1. Ghost Heat Footprint

As referenced earlier, we have made each ghost have its own `heat footprint` that extends outwards and incite pacman to run away. It builds upon the basic heuristic already implemented that used `manhattan distance` and gravely punished pacman if it was ever too close.

As shown in the code snippets below, the ghost use a `BFS` to extend their footprint. We have made this decision to better locate the ghost in relative position to pacman, as this way, walls give pacman more security and are respected by the heuristic. We also reduce the "level of danger" by each tile that the algorithm travels, because being 1 square away from death is certainly worse than having 3, although still not recommendable.

This heuristic has a lot of weight (-250 at the edge of the BFS) as is the primary way in which pacman maintains his life intact. Running away every chance someone gets near and is not scared is crucial for this strategy. It is also wary of when the `scare time` runs low, as it would not be pleasant to be trying to kill someone, just for them to turn around and end your life instead.

In [ ]:
# Factor 2: Proximidad a fantasmas
# Replaced by a heat footprint that follows the ghosts and better
# represent the dangerous level relative to pacman using a bfs strategy
for ghost_state in ghost_states:
    ghost_pos = ghost_state.getPosition()
    ghost_distance = manhattanDistance(pacman_pos, ghost_pos)

    # We maintain the urge to eat scared ghosts
    if ghost_state.scaredTimer > 0:
        # Si el fantasma está asustado, ir a por él
        score += 50 / (ghost_distance + 1)

    # We activate it moments before ghost activation to avoid
    # pacman running into them when they are going to return back
    elif ghost_state.scaredTimer > 37:
        updateGhostHeatMap(ghost_pos, layout, 3, 1000, heatmap)

    else:
        # Si no está asustado, evitarlo
        updateGhostHeatMap(ghost_pos, layout, 3, 1000, heatmap)



def updateGhostHeatMap(g_pos: tuple, layout: Grid, max_dist: int,
                       scare_points: int, ghost_hm):

    """
    Updates the heatmap to add the heat footprint of the ghosts. This footprint
    is just a bfs starting at the ghost position.

    This is a soft addition to the heatmap, so it can be done at any point.
    """
    gx, gy = int(g_pos[0]), int(g_pos[1])

    queue = Queue()

    queue.push((gx, gy, 0))
    ghost_hm[gx][gy] -= scare_points
    visited = {(gx, gy)}

    while not queue.isEmpty():
        x, y, d = queue.pop()

        nd = d + 1
        if nd >= max_dist:
            continue

        for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            nx = x + dx
            ny = y + dy

            if not layout[nx][ny] and (nx, ny) not in visited:
                center_distance = int(manhattanDistance((nx, ny), (gx, gy)))
                ghost_hm[nx][ny] -= scare_points / (center_distance + 1)
                queue.push((nx, ny, nd))
 
            visited.add((nx, ny))

### 1.2. Food Scent

The second most important heuristic is also built upon an already implemented heuristic, but in a much more impactful and complex way. This heuristic was also originally implemented using `manhattan distance` and had a very low reward (only 1 point divided by the distance to pacman). This implementation led to some troubles that we have mitigated giving it more relevance and presence.

Previously, pacman was not really chasing food, and was more occupied running for its life. Coming by some food certainly helped, but it was not in the slightest the primary goal of the yellow ball. It also got confused very easily by walls. The way it calculated distance did not account for walls and led to pacman thinking that it was closer to food than it really was. This lead to some situations where pacman would sit still or pacing back and forth trying to get a piece of food that he thought was a couple cells away, when in reality it was 7-8 movements of real distance.

We have solved this problem by implementing a circle of smell around pacman. This represents how far he can sense food. However, the way the distance is calculated is a little bit more complex than before. We calculate a `BFS` from every food piece in the circle, and give each food a movement capacity of double the radius of smell. This way, each food scent has an opportunity to reach pacman, even when it has to travel around walls. However, each food works individually and each cell only select the most powerful `scent`. This helps pacman reach the nearest food first and not get sidetracked by farther clusters of food.

Similar to the `heat footprint` of the ghosts, the `scent` of each piece of food starts at a 100 (to make it enticing) and gradually wears off the farther it travels towards our protagonist. However, there was a big problem with the implementation. As, once the food is collected, it no longer produced any type of scent, making it far more rewarding to stay close to the food instead of actually eating it. We solved this issue by punishing pacman for the number of food pieces still in the terrain. We kindly named it "hammering in nails to him until he picks up the fruit".

Finally, if there are little food left to pick up (< 15), his sense of smell becomes keener to have him sense the food from whichever corner he left them abandoned. This is needed as our implementation heavily relies on having fruit near, otherwise, it is left up to the ghosts to sheep him around until he smells another piece of fruit. 

Here is a code snippet that shows every reference to the `scent` functions:

In [ ]:
# Factor 1: Distancia a la comida más cercana
# Replaced manhattan distances by a sense of smell as the former would
# bypass walls and trick pacman
if food:
    # We "hammer in nails" to pacman if he does not pickup food
    score -= 150 * len(food)
    # Update the heatmap
    PacmanSmell(pacman_pos, layout, food, 5, 100, heatmap)
    

def circle(cx: int, cy: int, radio: int) -> list[tuple]:
    """
    Collects the points that form a circle with a certain radius and center
    """
    points = []
    r2 = radio**2

    for x in range(cx - radio, cx + radio + 1):
        for y in range(cy - radio, cy + radio + 1):
            if (x - cx) ** 2 + (y - cy) ** 2 <= r2:
                points.append((x, y))

    return points


def bfsSmell(layout, max_dist, food, px, py) -> list[tuple]:
    """
    Returns the shortest path between a piece of food and pacman's position
    """

    fx, fy = int(food[0]), int(food[1])
    queue = Queue()
    queue.push((fx, fy, 0, [(fx, fy, 0)]))
    visited = {(fx, fy)}
    
    while not queue.isEmpty():
        x, y, d, p = queue.pop()
        nd = d + 1

        if nd > 2*max_dist:
            continue

        for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            nx = x + dx
            ny = y + dy

            if not layout[nx][ny] and (nx, ny) not in visited:
                if (nx, ny) == (px, py):
                    return p + [(nx, ny, nd)]
                queue.push((nx, ny, nd, p + [(nx, ny, nd)]))
            visited.add((nx, ny))

    return []


def PacmanSmell(p_pos: tuple, layout: Grid, food_pos: list,
                max_dist: int, fruit_points: int, smell_hm):

    """
    Updates the heatmap to incorporate the smell of the closest food using
    a bfs strategy for every food in the circle and then selecting the best
    score for each cell

    This hard-sets the heatmap to the score, so this must be the first
    heuristic to apply
    """
    px, py = int(p_pos[0]), int(p_pos[1])

    food_dict = {}
    # If there is little food, we amp up massively the smell senses of pacman
    if len(food_pos) <= 15:
        max_dist = 100

    smell_area = circle(px, py, max_dist)
    for p in smell_area:
        if p in food_pos:
            bfs_path = bfsSmell(layout, max_dist, p, px, py)
            length = len(bfs_path)
            for x, y, d in bfs_path:
                points = fruit_points * (length - d) / length
                food_dict[(x, y)] = (points if (x, y) not in food_dict
                                     else max(points, food_dict[(x, y)]))

    for pos, points in food_dict.items():
        smell_hm[pos] = points
    
    return True if food_dict else False


### 1.3. Minor Heuristics

The next two heuristic are very little, but they each have helped pacman in some niche way to surpass their adversaries. 

#### 1.3.1. The Original HeatMap

At first, the `heatmap` was used as a way to map out the intersections and long corridors within the maze. As it is safer to stay in the intersections where the ghosts can not surround us.

This map is composed by a grid marked with numbers representing the distance of each cell to the closest intersection. This was made possible launching, yet again, the `BFS` algorithm from every point where more than three roads met and exploring until reaching other explorers or dead ends.

`BFS` was heavily used as it is the easiest and most modifiable search algorithm and always finds the best path, using low resources in such tiny maps and situations in which we are using them.

However, this heuristic is now deprecated and is used only as the base for our other more important heuristics. It has to be also said, that in the beginning, it helped pacman to stay away from corridors when ghosts where close.

Here is the heatmap creation and population snippet:

In [ ]:
# We create the heatmap only one time, as it is fixed
if self.heatmap is None:
    ghost_positions = []
    for g in ghost_states:
        x, y = g.getPosition()
        ghost_positions.append((int(x), int(y)))
    self.heatmap = createHeatMap(layout, ghost_positions)

# We make it local to add our heuristics
heatmap = deepcopy(self.heatmap)


def collectIntersections(layout: Grid) -> list[tuple[int, int]]:
    """
    Returns a list with the position of all the intersections in the map
    An intersection being a path with 3 or more movement possibilities
    """
    height = layout.height
    width = layout.width

    intersections = []

    for y in range(height):
        for x in range(width):
            # If there is a wall, skip
            if layout[x][y]:
                continue

            neighbors = 0
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nx = x + dx
                ny = y + dy
                # The layout is cornered by walls, so this function is never
                # going to exceed the limits
                if not layout[nx][ny]:
                    neighbors += 1

            if neighbors >= 3:
                intersections.append((x, y))
    return intersections


def createHeatMap(layout: Grid, ghost_position: list[tuple]):
    """
    Creates a heatmap with the level of danger of the tiles, this level being
    calculated by how far the tile is from an intersection. The intersections
    count as safe places because you can always scape from ghosts.
    """
    width = layout.width
    height = layout.height
    intersections = collectIntersections(layout)

    # We implement a BFS from every intersection
    heatmap = np.full((width, height), -1)
    queue = Queue()

    for x, y in intersections:
        heatmap[x][y] = 1  # 1 == safe
        queue.push((x, y, 1))  # x, y, distanceToIntersection

    while not queue.isEmpty():
        x, y, dist = queue.pop()
        ndist = dist + 1

        for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            nx = x + dx
            ny = y + dy

            if not layout[nx][ny] and heatmap[nx][ny] == -1:
                heatmap[nx][ny] = ndist
                queue.push((nx, ny, ndist))

    heatmap = initialCage(ghost_position, heatmap, layout)

    return heatmap

#### 1.3.2. The Cage Boundary

In each classical map of pacman, there is a cage from where every ghost spawns at the beginning and when they are killed. This tends to be a small void cage in which there is nothing and is preferable to not enter under any circunstances, as it normally leads to death.

We have made the decision to ban this tiles from pacman (unless certain death is upon him anyway), as he tended to wander in when he were harassed by a ghost and died earlier than needed. We solved this issue, by mapping the cage and covering its location with -99999 points. As all cages share similar structure (an exit wich leads to a corridor of some kind) it was easy to fill it up and look for intersections, as they were the indicator that the cage was over.

Here is the code snippet where we map it out:

In [ ]:
def initialCage(ghost_positions, heatmap, layout):
    """
    Locates the initial cage or home of the ghost. Every classical pacman
    map has a box or cage where the ghosts start and does not contain food
    and generally is a death sentence.
    """
    CAGE_POINTS = -99999
    queue = Queue()
    visited = set()

    # We locate ghost position
    for x, y in ghost_positions:
        x, y = int(x), int(y)

        queue.push((x, y))
        visited.add((x, y))
        heatmap[x][y] = CAGE_POINTS

    while not queue.isEmpty():
        x, y = queue.pop()

        for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            nx = x + dx
            ny = y + dy

            if layout[nx][ny]:
                continue

            if (nx, ny) in visited:
                continue

            is_exit = False

            for Dx, Dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                Nx = nx + Dx
                Ny = ny + Dy

                # If the neighbour of nx, ny is touching the exterior,
                # we do not expand further
                if heatmap[Nx][Ny] > 1 and heatmap[Nx][Ny] > CAGE_POINTS:
                    is_exit = True
                    break

            visited.add((nx, ny))

            # No expanding or scoring outside house
            if not is_exit:
                heatmap[nx][ny] = CAGE_POINTS
                queue.push((nx, ny))

    return heatmap

## 2. Task 2 - Train the model with good games

We have trained our `Neural Network` with 41 winning games. Of which, 23 where handmade and 17 where created by a combination of the `NeuralAgent` and the `AlphaBetaAgent` with all the heuristics. 

We have tried to avoid letting by games in which the strategy relied heavily in luck or did strange moves (staying still facing a wall, running around without a destiny in mind...).

If we had to make a ranking by how effective the games were, we had to rank `AlphaBeatAgent` as the absolut winner, as it had the most points in the game that it won, however, it was not as reliable as the human players, that accomplished more wins but with fewer points. The `NeuralAgent`, accomplished some winning games with average scores, but it was the fastest, by far, to execute.

## 3. Task 3 - Implement AlphaBetaAgent 

We have also implemented the `AlphaBetaAgent`, which uses alpha beta as its base to improve its performance in the game. It plays against itself for a couple of rounds (`depths`) and selects the best movement based on the best movements that the oponnents would chose in a perfect world.

It is computationally very expensive, but it shows its worth. In each turn that plays the pacman in his head (in alpha beta or min max, depends if we are pruning or not), it calls an evaluation function. This evaluation function determines how good a movement is. In our case, the evalutation function is the following:

In [ ]:
def evaluation_combined(self, state):
    # 1) Traditional score (with the new heuristics from Task 1)
    trad_score = self.traditional_evaluation(state)
    
    # 2) Neural network score
    neural_score = self.neural_evaluation(state)
    
    # 3) Weighted combination
    return self.w_heuristic * trad_score + self.w_neural * neural_score

This function calculates the score based on the heuristics and networks of the previous parts. On one hand, we have the `trad_score` which is composed by the handmade heuristics that we talked abour in the first point of the report. On the other hand, it is consider the `neural_score`, which emanates directly from the neural network trained with the previous winning games.

Each of this evaluations is calculated for each possible movement for each round and depth, which escalates very rapidly. We also have the possibility of choosing the weight each score has on the final result. We have opted for a 0.8 weight in heuristics and 0.2 in the neural network. As we are not as good pacman players as programmers, we trust more the heuristics that has been made, and leave only the neural weight to chip in when the heuristics are not enough.

## 4. Task 4 - Report results in a comparative table

### 4.1. Greedy neural agent (no modifications)

Classical layout:

In [3]:
!python pacman.py -p NeuralAgent -n 10

Modelo cargado correctamente desde models/pacman_model.pth
Tamaño de entrada: (20, 11)
NeuralAgent inicializado, usando dispositivo: cpu
Reply mode: False
Pacman died! Score: -265
Datos del juego 72 guardados en pacman_data/game_72.csv
Pacman died! Score: -87
Datos del juego 73 guardados en pacman_data/game_73.csv
Pacman died! Score: 67
Datos del juego 74 guardados en pacman_data/game_74.csv
Pacman died! Score: -35
Datos del juego 75 guardados en pacman_data/game_75.csv
Pacman died! Score: -79
Datos del juego 76 guardados en pacman_data/game_76.csv
Pacman died! Score: -32
Datos del juego 77 guardados en pacman_data/game_77.csv
Pacman died! Score: 89
Datos del juego 78 guardados en pacman_data/game_78.csv
Pacman died! Score: 144
Datos del juego 79 guardados en pacman_data/game_79.csv
Pacman died! Score: -261
Datos del juego 80 guardados en pacman_data/game_80.csv
Pacman died! Score: -277
Datos del juego 81 guardados en pacman_data/game_81.csv
Average Score: -73.6
Scores:        -265.0, 

New layout:

In [4]:
!python pacman.py -p NeuralAgent -n 10 -l customMaze

Modelo cargado correctamente desde models/pacman_model.pth
Tamaño de entrada: (20, 11)
NeuralAgent inicializado, usando dispositivo: cpu
Reply mode: False
Pacman died! Score: -314
Datos del juego 82 guardados en pacman_data/game_82.csv
Pacman died! Score: -393
Datos del juego 83 guardados en pacman_data/game_83.csv
Pacman died! Score: -57
Datos del juego 84 guardados en pacman_data/game_84.csv
Pacman died! Score: -53
Datos del juego 85 guardados en pacman_data/game_85.csv
Pacman died! Score: 287
Datos del juego 86 guardados en pacman_data/game_86.csv
Pacman died! Score: 135
Datos del juego 87 guardados en pacman_data/game_87.csv
Pacman died! Score: 536
Datos del juego 88 guardados en pacman_data/game_88.csv
Pacman died! Score: 276
Datos del juego 89 guardados en pacman_data/game_89.csv
Pacman died! Score: 507
Datos del juego 90 guardados en pacman_data/game_90.csv
Pacman died! Score: 214
Datos del juego 91 guardados en pacman_data/game_91.csv
Average Score: 113.8
Scores:        -314.0,

### 4.2. Greedy neural agent + 2 new heuristics

Classical layout:

In [2]:
!python pacman.py -p NeuralAgent -n 10

Modelo cargado correctamente desde models/pacman_model.pth
Tamaño de entrada: (20, 11)
NeuralAgent inicializado, usando dispositivo: cpu
Reply mode: False
Pacman died! Score: 92
Datos del juego 62 guardados en pacman_data/game_62.csv
Pacman died! Score: -383
Datos del juego 63 guardados en pacman_data/game_63.csv
Pacman died! Score: 196
Datos del juego 64 guardados en pacman_data/game_64.csv
Pacman died! Score: 464
Datos del juego 65 guardados en pacman_data/game_65.csv
Pacman emerges victorious! Score: 1384
Datos del juego 66 guardados en pacman_data/game_66.csv
Pacman died! Score: 232
Datos del juego 67 guardados en pacman_data/game_67.csv
Pacman died! Score: 362
Datos del juego 68 guardados en pacman_data/game_68.csv
Pacman died! Score: 134
Datos del juego 69 guardados en pacman_data/game_69.csv
Pacman emerges victorious! Score: 1718
Datos del juego 70 guardados en pacman_data/game_70.csv
Pacman died! Score: 469
Datos del juego 71 guardados en pacman_data/game_71.csv
Average Score: 

New layout:

In [5]:
!python pacman.py -p NeuralAgent -n 10 -l customMaze

Modelo cargado correctamente desde models/pacman_model.pth
Tamaño de entrada: (20, 11)
NeuralAgent inicializado, usando dispositivo: cpu
Reply mode: False
Pacman emerges victorious! Score: 1669
Datos del juego 92 guardados en pacman_data/game_92.csv
Pacman died! Score: 264
Datos del juego 93 guardados en pacman_data/game_93.csv
Pacman emerges victorious! Score: 1473
Datos del juego 94 guardados en pacman_data/game_94.csv
Pacman died! Score: 617
Datos del juego 95 guardados en pacman_data/game_95.csv
Pacman died! Score: -268
Datos del juego 96 guardados en pacman_data/game_96.csv
Pacman died! Score: -251
Datos del juego 97 guardados en pacman_data/game_97.csv
Pacman died! Score: 43
Datos del juego 98 guardados en pacman_data/game_98.csv
Pacman died! Score: 267
Datos del juego 99 guardados en pacman_data/game_99.csv
Pacman died! Score: 763
Datos del juego 100 guardados en pacman_data/game_100.csv
Pacman died! Score: 109
Datos del juego 101 guardados en pacman_data/game_101.csv
Average Sc

### 4.3. Alpha-Beta + network + heuristics (final)

Classical layout:

In [1]:
!python pacman.py -p AlphaBetaAgent -a evalFn=evaluation_combined,depth=3,w_heuristic="0.8",w_neural="0.2" -n 10

Reply mode: False
Pacman emerges victorious! Score: 1727
Datos del juego 52 guardados en pacman_data/game_52.csv
Pacman emerges victorious! Score: 1723
Datos del juego 53 guardados en pacman_data/game_53.csv
Pacman emerges victorious! Score: 1903
Datos del juego 54 guardados en pacman_data/game_54.csv
Pacman emerges victorious! Score: 1882
Datos del juego 55 guardados en pacman_data/game_55.csv
Pacman died! Score: -294
Datos del juego 56 guardados en pacman_data/game_56.csv
Pacman emerges victorious! Score: 1493
Datos del juego 57 guardados en pacman_data/game_57.csv
Pacman emerges victorious! Score: 1937
Datos del juego 58 guardados en pacman_data/game_58.csv
Pacman died! Score: 348
Datos del juego 59 guardados en pacman_data/game_59.csv
Pacman died! Score: 614
Datos del juego 60 guardados en pacman_data/game_60.csv
Pacman died! Score: -374
Datos del juego 61 guardados en pacman_data/game_61.csv
Average Score: 1095.9
Scores:        1727.0, 1723.0, 1903.0, 1882.0, -294.0, 1493.0, 1937.

New layout:

In [6]:
!python pacman.py -p AlphaBetaAgent -a evalFn=evaluation_combined,depth=3,w_heuristic="0.8",w_neural="0.2" -n 10 -l customMaze

Reply mode: False


### 4.4. Report Table

|Configuration|Classical Layout|New Layout|
|-|-|-|
|Greedy neural agent (no modifications)|-73.6 / 0%|113.8 / 0%|
|Greedy neural agent + 2 new heuristics|466.8 / 20%|468.6 / 20%|
|Alpha-Beta + network + heuristics (final)|1095.9 / 60%|B|


As we can see, adding the heuristics increased massively the average performance of the Agent. However, it did not make its win percentage go extremely high, but its survivability did increase greatly.

On the other hand, adding Alpha-Beta to the mix really amped up the win rate and average performance. With its abbility to preview enemy movements rounds before it happens, it can anticipate